In [2]:
import pandas as pd 
import sys 
import numpy as np 
sys.path.append('/home/joag/2024-ESL-BiomassSAR')
from src.fdlbiomass.dataloader_fns import generate_quadrants, generate_dataloaders

In [3]:
from src.fdlbiomass.unet_model import UNet3D

In [4]:
model = UNet3D(in_channels=2) 

In [5]:
import torch
from torchvision import models
from torchsummary import summary

# pretrainedmodel_vgg = models.vgg16()
# summary(pretrainedmodel_vgg, (3, 224, 224))

In [6]:
print('\n\nModel params:')
for param in model.parameters():
    print(param)



Model params:
Parameter containing:
tensor([[[[[ 0.0059, -0.1050,  0.0579],
           [-0.0326, -0.0473, -0.0169],
           [-0.0088,  0.0134, -0.0286]],

          [[ 0.0913, -0.0315, -0.0567],
           [ 0.1084, -0.1302, -0.0990],
           [-0.0739, -0.0611, -0.0709]],

          [[ 0.0489, -0.0203, -0.0318],
           [-0.0040, -0.1113,  0.0321],
           [-0.0764, -0.0766, -0.0110]]],


         [[[-0.1093,  0.1290, -0.0898],
           [ 0.0290, -0.0230,  0.1235],
           [ 0.0160,  0.0247,  0.0572]],

          [[ 0.0823, -0.0026,  0.0690],
           [-0.0326,  0.0925,  0.1346],
           [-0.0806, -0.1309,  0.1036]],

          [[-0.0285,  0.0596, -0.0009],
           [-0.1116,  0.0732,  0.1190],
           [ 0.0395, -0.0237,  0.1303]]]],



        [[[[-0.0608, -0.0416, -0.0074],
           [ 0.1134,  0.0050,  0.0065],
           [ 0.1261,  0.0803, -0.0546]],

          [[ 0.0894, -0.1185, -0.0841],
           [ 0.0128, -0.1353,  0.0135],
           [-0.1302,  

In [10]:
summary(model, (2, 16,16,36))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1       [-1, 64, 16, 16, 36]           3,520
       BatchNorm3d-2       [-1, 64, 16, 16, 36]             128
              ReLU-3       [-1, 64, 16, 16, 36]               0
            Conv3d-4       [-1, 64, 16, 16, 36]         110,656
       BatchNorm3d-5       [-1, 64, 16, 16, 36]             128
              ReLU-6       [-1, 64, 16, 16, 36]               0
        DoubleConv-7       [-1, 64, 16, 16, 36]               0
         MaxPool3d-8         [-1, 64, 8, 8, 36]               0
            Conv3d-9        [-1, 128, 8, 8, 36]         221,312
      BatchNorm3d-10        [-1, 128, 8, 8, 36]             256
             ReLU-11        [-1, 128, 8, 8, 36]               0
           Conv3d-12        [-1, 128, 8, 8, 36]         442,496
      BatchNorm3d-13        [-1, 128, 8, 8, 36]             256
             ReLU-14        [-1, 128, 8

In [3]:
parent_path = '/home/grace.bcolverd/esl_dev/tomosar-matlab'
polarimetric_channel=['HV', 'VV', 'VV']
q, gt = generate_quadrants(parent_path, polarimetric_channels=polarimetric_channel  )

Original channels: ['HV', 'VV', 'VV']
Unique channels: ['VV', 'HV']
Starting to load the biomass data
Load complete
Original shape: (326, 840)
Cropped shape: (321, 665)
665 665


In [ ]:
q[0].shape, gt[0].shape

((160, 332, 36, 3), (160, 332))

In [11]:
def check_channel_correlation(dataloader):
    all_correlations = []
    for inputs, _ in dataloader:
        for i in range(inputs.shape[1]):
            for j in range(i+1, inputs.shape[1]):
                corr = np.corrcoef(inputs[:, i, ...].flatten(), inputs[:, j, ...].flatten())[0, 1]
                all_correlations.append(corr)
    
    print(f"Average channel correlation: {np.mean(all_correlations):.4f}")
    print(f"Min correlation: {np.min(all_correlations):.4f}")
    print(f"Max correlation: {np.max(all_correlations):.4f}")



In [14]:
train_loader, test_loader, val_loader, dataset = generate_dataloaders(q, gt, (64,64,36), (32,32,36), test_quadrant=3, batch_size=4)

for id, item in train_loader:
    print(id.shape)
    print(item.shape)

#     torch.Size([4, 2, 64, 64, 36])
# torch.Size([4, 64, 64])

torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([2, 2, 64, 64, 36])
torch.Size([2, 64, 64])


In [15]:
check_channel_correlation(train_loader)

Average channel correlation: 0.8489
Min correlation: 0.8190
Max correlation: 0.9793


In [5]:
# Example usage



dataset.set_mode('train')
print(len(dataset))  # Number of training patches
print(dataset[0][0].shape)  # First patch and its ground truth


72
torch.Size([2, 64, 64, 36])


In [8]:
for id, item in t:
    print(id.shape)
    print(item.shape)

torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
torch.Size([4, 2, 64, 64, 36])
torch.Size([4, 64, 64])
